In [33]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
from dotenv import load_dotenv
load_dotenv(override=True)

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

In [35]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [36]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [42]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [38]:
def run_test_case(test_case): 
  output = run_prompt(test_case) 

  # Grade the output 
  model_grade = grade_by_model(test_case, output) 
  score = model_grade["score"] 
  reasoning = model_grade["reasoning"] 

  return { 
    "output": output, 
    "test_case": test_case, 
    "score": score, 
    "reasoning": reasoning 
  }


In [39]:
from statistics import mean 

def run_eval(dataset): 
  results = [] 

  for test_case in dataset: 
    result = run_test_case(test_case) 
    results.append(result) 

  average_score = mean([result["score"] for result in results]) 

  print(f"Average score: {average_score}") 

  return results

In [43]:
import json

with open("lesson11-dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8


In [44]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Looking at this task, I need to parse an AWS ARN (Amazon Resource Name) and extract the service component.\n\nAWS ARNs follow this general format:\n```\narn:partition:service:region:account-id:resource-id\n```\n\nThe service name is always the 3rd component (index 2) when split by colons.\n\nHere's my solution:\n\n```python\ndef extract_service_from_arn(arn):\n    \"\"\"\n    Extracts the service name from an AWS ARN string.\n    \n    Args:\n        arn (str): AWS ARN string (e.g., 'arn:aws:s3:::my-bucket')\n    \n    Returns:\n        str: The service name (e.g., 's3')\n    \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    parts = arn.split(':')\n    \n    # ARN must have at least 6 parts (some fields can be empty)\n    if len(parts) < 6:\n        raise ValueError(f\"Invalid ARN format: {arn}\")\n    \n    # Validate it s